# Neutral atom hardware quickstart

这个 notebook 是控制电脑上的硬件流程：连接 qCMOS 和 FPGA sequencer，配置
pulse sequence，拍 raw image，校准 sitemap 和 threshold，detect，最后扫
detection time。

运行前先在 Verilog/FPGA 电脑上启动 sequencer server：

```powershell
cd "D:\ZLC"
.\fpga\build_and_program.bat --check
.\fpga\build_and_program.bat
.\fpga\run_server.bat --check-config
.\fpga\run_server.bat
```

默认硬件路线是 address-switch pulse streamer。server 从 XDC 推断完整
channel order；GUI 或 API 可以只显示/配置其中几路，但上传时会自动补成
full-width mask，没配置的 channel 全部为 off。默认相机成像子集是：

```text
ch09 trap
ch00 cooling
ch03 probe
ch11 emCCD
```

The same XDC also has `ch06 trig`, but the checked-in camera preset uses
`ch11/emCCD/M13` as the qCMOS/emCCD trigger.

默认 clock 是 50 MHz，也就是 20 ns step。所有 duration、delay、scan x/y
值都必须对齐到这个 step。

In [ ]:
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
sys.path.insert(0, PROJECT_ROOT)
os.environ["PYTHONPATH"] = PROJECT_ROOT

import Zou_lab_control.frontend as zf

zf.notebook_setup()

In [ ]:
from pathlib import Path
import numpy as np

import Zou_lab_control.frontend as zf
import Zou_lab_control.neutral_atom as na

try:
    zf.use_widget_backend()
except Exception as exc:
    print(f"Widget backend not enabled here: {exc}")

zf.enable_long_output()
zf.apply_style()

## Connect hardware

`na.connect(..., open_devices=True)` 会通过 device loader 构造、校验并打开
camera/sequencer。把 `host` 改成 FPGA/Vivado 电脑的 IP。

In [ ]:
exp = na.connect(
    "remote_template",
    sequencer={"host": "192.168.0.20", "port": 18861},
    open_devices=True,
)

# First-light manual trigger path:
# exp = na.connect("manual_template", open_devices=True)

exp

## Configure and preflight the imaging sequence

`PulseSequence` 是 hardware 和 notebook 共同使用的时序源。address-switch
sequencer 会把 imaging helper 映射到 `ch09/ch00/ch03/ch11`。通过
`preflight.raise_if_failed()` 之后再拍照。

In [ ]:
exp.timing.configure_imaging(exposure=2e-3, load=True, trigger_width=20e-6, pre_trigger=100e-6)
pulse_plot = exp.timing.plot_sequence()
preflight = exp.timing.preflight()
preflight.summary()

In [ ]:
preflight.raise_if_failed()

## Optional: edit pulses with the PyQt pulse GUI

GUI 只是 pulse 前端。它读取 `exp.devices.sequencer.channels`，编辑
`PulseTableState`，然后在 `On Pulse/Stop Pulse` 按钮里调用同一个
sequencer。`On Pulse` 会先把当前 pulse state 上传到 sequencer，再立刻
start；`Stop Pulse` 调用 safe/reset。

如果当前环境没有桌面/Qt，跳过这个 cell，继续用
`exp.timing.configure_imaging(...)` 和 API 配置 pulse。

In [ ]:
# Uncomment on a desktop Python/Qt environment.
# pulse_gui = zf.show_pulse_gui(
#     experiment=exp,
#     state=na.PulseTableState.load("pulses/camera_imaging_address_switch.json"),
#     scale=0.82,
#     window_ratio=0.90,
# )
# pulse_gui

## Capture a camera image

`capture` 只显示 raw camera frame；site overlay 只属于
calibration/readout/detect 图。

In [ ]:
capture = exp.camera.capture(frames=1, display=True)
capture.summary()

## Calibrate sitemap

hardware config 没有 virtual `trap_array`，所以 sitemap 需要显式给出 site grid。

In [ ]:
grid_shape = (5, 7)
sitemap = exp.readout.sitemap(frames=20, grid_shape=grid_shape, roi_radius=1, display=True)
sitemap.summary()

## Calibrate thresholds

threshold calibration 依赖刚刚得到的 sitemap。

In [ ]:
threshold = exp.readout.thresholds(frames=120, site=0, display=True)
threshold.summary()

## Detect one shot

`DetectionResult.occupied` 是后续 rearrangement/statistics 可以直接使用的
boolean array。

In [ ]:
shot = exp.readout.detect(display=True)
occupancy_grid = shot.occupied.reshape(grid_shape)
occupancy_grid, shot.summary()

## Bind a pulse for x/y scans

对于 readout-time 或曝光宽度扫描，可以把一张 `PulseTableState` 绑定到当前
session 的 sequencer。仓库里的
`pulses/camera_imaging_address_switch.json` 已经把 `camera_exposure` period
写成 `duration="x", unit="str (ns)"`，默认 `x_ns=19_980_000`，所以
`pulse.x` 就是 probe/readout exposure 的 ns 数。

GUI/API 的 scan array 是一个字段；GUI 左侧会随开关显示 `Scan X` 或
`Scan XY`：

```text
Use Y off / Scan X:  [x0, x1, ...]
Use Y on  / Scan XY: [(x0, y0), (x1, y1), ...]
```

所有值是 ns，必须对齐到 20 ns。Preview 不展开所有 scan points，而是把
包含 `x`、`y` 或 `100000-x` 这类表达式的时间段标出来。

传给 camera acquisition 时，`exp.readout.detection_time(..., pulse=pulse)`
会用同一张 pulse 先拍 long-reference，再为每个扫描点临时生成刚好 `shots`
个外部触发的有限序列，保证相机先 arm，再由同一个 sequencer fire。

In [ ]:
pulse = exp.timing.bind_pulse("pulses/camera_imaging_address_switch.json")
pulse.snapshot()

# This does not fire hardware; it shows that x controls the finite readout
# sequence duration before you run the scan.
test_widths_ns = [2_000_000, 4_000_000, 8_000_000]
[(width, pulse.frame_sequence(1, x_ns=width).duration) for width in test_widths_ns]

RUN_SINGLE_PULSE_TEST = False

single_program = None
if RUN_SINGLE_PULSE_TEST:
    pulse.x = 2_000_000  # ns
    single_program = pulse.on_pulse(wait=True, timeout=10.0, repeat_forever=False)
single_program

# Free-running output is still explicit when you want it:
# pulse.on_pulse(wait=False, repeat_forever=True)

## Scan detection time and fidelity

这个 scan 使用 camera images，不使用任何 ground truth。第一次上机默认同步跑完；
确认流程稳定后，下一格有一个显式的 live 版本：只把
`RUN_LIVE_READOUT_SCAN` 改成 `True`，其它 API 形状不变，仍然通过同一个
`pulse` 和 remote sequencer。

In [ ]:
clock_hz = exp.devices.sequencer.clock_hz
time_ticks = np.linspace(int(round(0.2e-3 * clock_hz)), int(round(8e-3 * clock_hz)), 40, dtype=int)
times = time_ticks / clock_hz
scan = exp.readout.detection_time(times, shots=30, live=False, display=True, pulse=pulse)
fit_result, popt = scan.data_figure.decay(is_display=False)
scan.summary(), fit_result, popt

## Optional live readout-time scan

这个 cell 是控制电脑上最短的 live readout-time/fidelity 工作形状。它不会改
FPGA 电脑的 server，也不需要重新打开 GUI。

In [ ]:
RUN_LIVE_READOUT_SCAN = False

live_scan = None
if RUN_LIVE_READOUT_SCAN:
    live_scan = exp.readout.detection_time(times, shots=30, live=True, display=True, pulse=pulse)
live_scan